# Microsoft Foundry · Criando um agente de IA com Python

Laboratório do curso de **Microsoft Foundry** (Seção 4 · Aula 13): criação de um **agente de IA** inteiramente pelo código, usando o **Foundry SDK** (`azure-ai-projects`), e conversa com ele mantendo a **memória** entre as mensagens por meio de uma *conversation*.

O agente criado é o **Batman** 🦇: ele responde a qualquer pergunta no personagem do Cavaleiro das Trevas de Gotham City.

## Sumário

0. [Visão geral do projeto](#0.-Visão-geral-do-projeto)
1. [Instalação das bibliotecas](#1.-Instalação-das-bibliotecas)
2. [Variáveis de ambiente](#2.-Variáveis-de-ambiente)
3. [Conectando ao projeto do Foundry](#3.-Conectando-ao-projeto-do-Foundry)
4. [Criando o agente Batman](#4.-Criando-o-agente-Batman)
5. [Criando a conversa (memória do agente)](#5.-Criando-a-conversa-(memória-do-agente))
6. [Conversando com o agente](#6.-Conversando-com-o-agente)
7. [Verificando no portal do Foundry](#7.-Verificando-no-portal-do-Foundry)
8. [Resumo dos aprendizados](#8.-Resumo-dos-aprendizados)
9. [Limpeza e custos](#9.-Limpeza-e-custos)
10. [Interface com Streamlit](#10.-Interface-com-Streamlit)

---
## 0. Visão geral do projeto

### 0.1 Problema

Criar um assistente de IA "do zero" exige muita coisa além do modelo: guardar o histórico das conversas, controlar versões do comportamento do assistente, autenticar com segurança e acompanhar quanto cada conversa consome. Fazer tudo isso manualmente dá trabalho e é fácil errar.

### 0.2 Objetivo

Usar o **Microsoft Foundry Agent Service** para que a Azure cuide dessa parte e o código fique simples:

- **Criar um agente** com personalidade definida (o Batman) a partir de um modelo GPT-4, direto do Python.
- **Versionar** o agente: cada nova criação gera uma versão (v1, v2, …).
- **Conversar** com o agente com **memória**, usando o `conversation.id`, de forma que ele lembre o que foi dito antes.
- **Acompanhar** no portal o agente criado e o consumo de tokens de cada conversa (*tracing*).
- **Levar para uma interface** de chat com Streamlit.

### 0.3 Ferramentas e serviços da Azure

| Serviço / ferramenta | Papel no projeto |
| --- | --- |
| **Microsoft Foundry** (projeto) | Plataforma que reúne modelos, agentes, conversas e observabilidade em um só lugar. |
| **Foundry Agent Service** | Hospeda o agente `batman-agent`, com instruções e versões. |
| **Azure OpenAI (deployment GPT-4)** | Modelo de linguagem que gera as respostas do agente. |
| **Microsoft Entra ID** | Identidade: autentica o código sem chave escrita no notebook (`DefaultAzureCredential`). |
| **Conversations (Responses API)** | Guarda o histórico de mensagens de cada conversa. |
| **Tracing do Foundry** | Mostra entradas, saídas e tokens de cada chamada. |
| `azure-ai-projects` 2.0.0b2 | **Foundry SDK** (versão 2): conecta o Python ao projeto e cria o agente. |
| `azure-identity` | Fornece o `DefaultAzureCredential`. |
| `openai` | Cliente usado para criar conversas e enviar mensagens ao agente. |
| `python-dotenv` | Lê as configurações do arquivo `.env`. |
| `streamlit` | Interface de chat (seção 10). |

### 0.4 Arquitetura

![Arquitetura do projeto](arquitetura.svg)

| Passo | O que acontece | Onde no código |
| --- | --- | --- |
| 1 | O código se autentica no **Entra ID** usando o login já feito na máquina. | `DefaultAzureCredential()` (seção 3) |
| 2 | O agente **batman-agent** é registrado no Agent Service como uma nova versão. | `project_client.agents.create_version` (seção 4) |
| 3 | Uma **conversa** é criada; o `conversation.id` vai guardar o histórico. | `openai_client.conversations.create()` (seção 5) |
| 4 | Cada pergunta é enviada à conversa, apontando para o agente. | `openai_client.responses.create(...)` (seção 6) |
| 5 | O agente envia **instruções + histórico + pergunta** ao GPT-4 e devolve a resposta. | feito pela Azure |
| 6 | Cada chamada fica registrada no **tracing** do portal, com os tokens. | portal do Foundry (seção 7) |

> 💡 **Para um leigo:** pense no agente como um **funcionário** contratado na Azure. O *modelo* (GPT-4) é o cérebro dele, as *instruções* são o manual de conduta ("aja como o Batman") e a *conversa* é o caderno onde ele anota tudo o que você já disse.

---
## 1. Instalação das bibliotecas

| Biblioteca | Para que serve |
| --- | --- |
| `azure-ai-projects==2.0.0b2` | **Foundry SDK v2**: o cliente do projeto (`AIProjectClient`) e a criação de agentes. Versão *beta* (`b2`), fixada para o código funcionar igual ao da aula. |
| `openai==1.109.1` | Cliente OpenAI, usado para criar conversas e obter as respostas do agente. |
| `python-dotenv` | Lê o arquivo `.env` com as configurações. |
| `azure-identity` | Autenticação na Azure (`DefaultAzureCredential`). |

> ⚠️ Versões fixadas (`==`) evitam surpresas: SDKs em *beta* mudam de nome de método entre versões.

In [ ]:
# Instalando as bibliotecas do projeto no mesmo ambiente do kernel do notebook (%pip)
# azure-ai-projects = Foundry SDK | openai = conversa com o agente
# python-dotenv = lê o .env | azure-identity = login na Azure sem chave no código
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

---
## 2. Variáveis de ambiente

O código precisa de duas informações do seu projeto no Foundry, guardadas no arquivo **`.env`** (fora do código):

| Variável | O que é | Onde encontrar no portal do Foundry |
| --- | --- | --- |
| `FOUNDRY_PROJECT_ENDPOINT` | Endereço (URL) do projeto Foundry. | Página **Início** (*Overview*) do projeto → *Project endpoint*. |
| `MODEL_DEPLOYMENT_NAME` | Nome do deployment do modelo (aqui, GPT-4). | **Modelos + endpoints** (*Models*) → nome do deployment. |

Formato do arquivo `.env` (na mesma pasta do notebook):

```
FOUNDRY_PROJECT_ENDPOINT=https://<seu-recurso>.services.ai.azure.com/api/projects/<seu-projeto>
MODEL_DEPLOYMENT_NAME=<nome-do-deployment>
```

> ⚠️ O `load_dotenv()` procura um arquivo chamado exatamente **`.env`**. O `env.txt` desta pasta serve só como modelo: copie (ou renomeie) para `.env` e preencha os valores.
>
> 🔒 O `.env` guarda apenas **endereços e nomes** (nenhuma chave), porque a autenticação é feita pelo Microsoft Entra ID (seção 3). Se algum dia ele passar a ter segredos (chaves de API, tokens), não deve ser publicado: nesse caso, adicione `.env` ao `.gitignore`.

In [ ]:
# Importando as bibliotecas e lendo as configurações do arquivo .env
# os.getenv busca cada valor pelo nome; se a variável não existir, retorna None
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

---
## 3. Conectando ao projeto do Foundry

O **`AIProjectClient`** é a "porta de entrada" do Python para o projeto no Foundry: é por ele que criamos agentes, consultamos conexões e obtemos o cliente OpenAI.

Ele precisa de duas coisas:

- **`endpoint`**: o endereço do projeto (do `.env`).
- **`credential`**: quem está acessando. O **`DefaultAzureCredential`** tenta, em ordem, várias formas de login já existentes na máquina e usa a primeira que funcionar:

| Ordem (resumida) | Fonte da credencial |
| --- | --- |
| 1 | Variáveis de ambiente de um *service principal* |
| 2 | *Managed Identity* (quando o código roda dentro da Azure) |
| 3 | Login da extensão Azure no **VS Code** |
| 4 | Login da **Azure CLI** (`az login`) |

> 💡 Assim não existe nenhuma chave escrita no código. Para rodar localmente, basta executar `az login` no terminal antes. O usuário precisa ter permissão no projeto (por exemplo, a função **Azure AI User**).

In [ ]:
# Criando o cliente do projeto Foundry: endpoint do .env + login da máquina (Entra ID)
# A conexão é "preguiçosa": erros de login só aparecem na primeira chamada real
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

---
## 4. Criando o agente Batman

Um agente no Foundry é definido por um **`PromptAgentDefinition`**, com dois itens principais:

| Campo | Valor | Significado |
| --- | --- | --- |
| `model` | `MODEL_DEPLOYMENT_NAME` | O "cérebro" do agente: o deployment do GPT-4. |
| `instructions` | `"You are Batman, the Dark Knight of Gotham City..."` | O *system prompt*: o manual de comportamento. Em português: *"Você é o Batman, o Cavaleiro das Trevas de Gotham City. Responda a todas as perguntas no personagem, como o Batman faria."* |

O método **`agents.create_version`** cria o agente com o nome `batman-agent`:

- Na primeira execução, gera a **versão 1**.
- **Toda vez que a célula é executada de novo**, é criada uma **nova versão** (2, 3, …) com o mesmo nome. Isso permite evoluir as instruções sem perder o histórico de versões.

> 💡 O modelo base não importa muito: a aula comenta que também daria para usar outro modelo do catálogo (como o Claude). Porém, a etapa de conversa (seção 5) usa o cliente **OpenAI**, que funciona com os modelos GPT.

In [ ]:
# Criando (ou criando uma nova versão de) o agente "batman-agent" no Foundry Agent Service
# PromptAgentDefinition = modelo (deployment GPT-4) + instruções (system prompt do Batman)
# Cada execução desta célula gera uma nova versão do agente
from azure.ai.projects.models import PromptAgentDefinition

agent_name = "batman-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are Batman, the Dark Knight of Gotham City. Respond to all queries in character as Batman would.",
    ),
)

# printing the agent id
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

O retorno traz o **id**, o **nome** e a **versão** do agente. O mesmo agente já aparece no portal do Foundry (seção 7).

---
## 5. Criando a conversa (memória do agente)

Para conversar com o agente usamos o **cliente OpenAI** obtido do próprio projeto (`project_client.get_openai_client()`). Ele já vem configurado com o endpoint e a autenticação do Foundry, sem chave extra.

Com ele criamos uma **conversation**: um objeto na Azure que **armazena o histórico** de mensagens entre o usuário e o agente. Ela é identificada por um `conversation.id`.

![Como a conversa dá memória ao agente](conversa_memoria.svg)

| Sem `conversation.id` | Com `conversation.id` |
| --- | --- |
| Cada pergunta chega sozinha ao modelo. | As mensagens anteriores são enviadas junto com a nova pergunta. |
| O agente não lembra o que você disse. | O agente lembra (por exemplo, o seu nome). |
| Menos tokens por chamada. | Os tokens **crescem** a cada mensagem. |

In [ ]:
# Obtendo o cliente OpenAI já autenticado no projeto Foundry
# e criando a conversa que vai guardar o histórico (memória) do chat
# creating an openai client first
openai_client = project_client.get_openai_client()

# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

---
## 6. Conversando com o agente

O chat é um laço `while` que fica aberto até o usuário digitar **`exit`** ou **`quit`**. A cada mensagem, chamamos **`openai_client.responses.create`** com:

| Parâmetro | Valor | Para que serve |
| --- | --- | --- |
| `conversation` | `conversation.id` | Grava a mensagem na conversa e envia o histórico ao modelo (memória). |
| `extra_body` → `agent` | `{"name": "batman-agent", "type": "agent_reference"}` | Diz **qual agente** responde. Sem informar a versão, é usada **a mais recente**. |
| `input` | texto digitado | A pergunta do usuário. |

A resposta em texto fica em **`response.output_text`**.

> ⚠️ No VS Code, o `input()` abre uma caixa no topo do editor. Digite `exit` para encerrar o chat; enquanto o laço estiver aberto, a célula continua em execução.

In [ ]:
# Laço de chat: envia cada pergunta ao agente Batman dentro da mesma conversa
# "exit" ou "quit" encerram o chat
chat = True

while chat:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        chat = False
        print("Exiting chat. Goodbye!")
    else:
        response = openai_client.responses.create(
            conversation=conversation.id,
            extra_body = {
                "agent": {
                    "name": agent_name,
                    "type": "agent_reference"
                }
            },
            input = user_input
        )

        print(f"Batman: {response.output_text}")

### 6.1 O que aconteceu na demonstração da aula

| Pergunta | Resposta do Batman (resumo) |
| --- | --- |
| "Meu nome é Kuljot Singh Bakshi. Como está Gotham?" | *"Gotham é a mesma de sempre: sombria, perigosa e no limite. A corrupção está em cada esquina… mas enquanto eu estiver aqui, os criminosos vão pensar duas vezes. Gotham pode estar quebrada, mas é a minha cidade."* |
| "Você pode me dizer qual era o meu nome?" | *"Não tenho superpoderes como o Super-Homem… trabalho com os fatos. Você me disse que seu nome era Kuljot Singh Bakshi. Nomes têm poder: honre o seu."* |

✅ Na segunda pergunta o agente **lembrou o nome** dito na primeira. Isso só é possível porque as duas mensagens foram enviadas na **mesma conversa** (`conversation.id`).

---
## 7. Verificando no portal do Foundry

Tudo o que foi criado pelo código também aparece no portal do **Foundry** (e no `portal.azure.com`):

- **Agentes** → `batman-agent` → **versão 1** (e as próximas, se a célula da seção 4 for executada de novo).
- **Traces** (rastreamento) → a conversa, com a **entrada e a saída** de cada mensagem e o consumo de **tokens**.

| Chamada | Tokens |
| --- | --- |
| 1ª pergunta | 139 |
| Conversa completa (2 perguntas) | 219 |

> 💡 **Inflação de tokens:** a segunda chamada consumiu mais porque o **histórico inteiro** é reenviado ao modelo. Em sistemas em escala, isso precisa ser gerenciado (por exemplo, limitando o tamanho do histórico, resumindo mensagens antigas ou iniciando novas conversas), pois tokens = custo e latência.

---
## 8. Resumo dos aprendizados

- O **Microsoft Foundry** reúne modelos, agentes, conversas e observabilidade em um único projeto, acessado pelo **Foundry SDK** (`azure-ai-projects`).
- O **`AIProjectClient`** conecta o Python ao projeto usando o **endpoint** e o **`DefaultAzureCredential`** (Entra ID), sem nenhuma chave no código.
- Um agente é **modelo + instruções** (`PromptAgentDefinition`). O `create_version` cria uma **nova versão** a cada execução.
- O **cliente OpenAI** do projeto cria **conversas**; o `conversation.id` dá **memória** ao agente.
- O `agent_reference` aponta para o agente pelo nome e, sem versão informada, usa a **mais recente**.
- O **tracing** do portal mostra entradas, saídas e tokens: o histórico faz o consumo de tokens **crescer** a cada mensagem.

---
## 9. Limpeza e custos

| Recurso | Gera custo? |
| --- | --- |
| Chamadas ao modelo (tokens) | Sim, **por uso**: só quando você conversa com o agente. |
| Agente e versões no Agent Service | A definição do agente em si não é a parte cara; o custo vem das chamadas ao modelo. |
| Deployment do modelo | Em deployments *Standard*/*Global Standard*, a cobrança é por token. Deployments com capacidade provisionada (PTU) cobram **por existir**. |

Para não deixar nada para trás ao terminar os estudos:

- **Portal do Foundry → Agentes** → apagar as versões do `batman-agent` que não serão mais usadas.
- **Modelos + endpoints** → apagar o deployment, se ele não for usado em outros labs.
- Ao final do curso, apagar o **grupo de recursos** inteiro no `portal.azure.com` remove tudo de uma vez.

> 💡 Vale criar um **orçamento com alerta** em *Cost Management → Budgets* para ser avisado de qualquer gasto inesperado.

---
## 10. Interface com Streamlit

A célula abaixo gera o arquivo **`app.py`**: o mesmo agente Batman, agora com uma interface de chat no navegador. Ela reaproveita o código do notebook:

| Função do `app.py` | Código do notebook reaproveitado |
| --- | --- |
| `criar_clientes()` | `AIProjectClient` + `DefaultAzureCredential` (seção 3) e `get_openai_client()` (seção 5) |
| `publicar_agente()` | `agents.create_version` com `PromptAgentDefinition` (seção 4) |
| `nova_conversa()` | `conversations.create()` (seção 5) |
| `perguntar()` | `responses.create` com `agent_reference` (seção 6) |

Diferenças importantes em relação ao notebook:

- O Streamlit **reexecuta o script inteiro** a cada interação. Por isso os clientes ficam em cache (`@st.cache_resource`) e o `conversation.id` fica no `st.session_state`. Sem isso, cada mensagem criaria uma conversa nova e o agente perderia a memória.
- O agente **não é recriado** a cada execução: o app usa o `batman-agent` já criado na seção 4. Para mudar as instruções, há um botão na barra lateral que publica uma **nova versão**.

**Como executar:**

```bash
pip install streamlit
az login
streamlit run app.py
```

O app abre em `http://localhost:8501` e usa o mesmo arquivo `.env` desta pasta.

In [ ]:
%%writefile app.py

import os

import streamlit as st
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.core.exceptions import ClientAuthenticationError, HttpResponseError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from openai import APIError

# ------------------------------------------------------------------
# 1. Configurações (as mesmas do notebook)
# ------------------------------------------------------------------
# O .env fica na mesma pasta do app.py (fora do código)
load_dotenv()

FOUNDRY_PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
AGENT_NAME = "batman-agent"
INSTRUCOES_PADRAO = (
    "You are Batman, the Dark Knight of Gotham City. "
    "Respond to all queries in character as Batman would."
)

st.set_page_config(
    page_title="Batman Agent · Microsoft Foundry",
    page_icon="🦇",
    layout="wide"
)


# ------------------------------------------------------------------
# 2. Conexão com o projeto do Foundry
# ------------------------------------------------------------------
# O Streamlit reexecuta o script a cada interação. O cache_resource
# faz os clientes serem criados uma única vez e reaproveitados.
@st.cache_resource
def criar_clientes():
    # Mesmo código da seção 3: endpoint do .env + login da máquina (az login)
    project_client = AIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT,
        credential=DefaultAzureCredential(),
    )
    # Mesmo código da seção 5: cliente OpenAI já autenticado no projeto
    openai_client = project_client.get_openai_client()
    return project_client, openai_client


# ------------------------------------------------------------------
# 3. Agente: publicar uma nova versão (só quando o usuário pedir)
# ------------------------------------------------------------------
def publicar_agente(instrucoes):
    project_client, _ = criar_clientes()
    # Mesmo código da seção 4: cada chamada cria uma nova versão do agente
    return project_client.agents.create_version(
        agent_name=AGENT_NAME,
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT_NAME,
            instructions=instrucoes,
        ),
    )


# ------------------------------------------------------------------
# 4. Conversa: a memória do agente
# ------------------------------------------------------------------
def nova_conversa():
    _, openai_client = criar_clientes()
    # Nova conversation = histórico zerado; o id fica no session_state
    conversation = openai_client.conversations.create()
    st.session_state.conversation_id = conversation.id
    st.session_state.messages = []


# ------------------------------------------------------------------
# 5. Pergunta ao agente (mesmo responses.create do laço de chat)
# ------------------------------------------------------------------
def perguntar(texto):
    _, openai_client = criar_clientes()
    response = openai_client.responses.create(
        conversation=st.session_state.conversation_id,
        extra_body={
            "agent": {
                "name": AGENT_NAME,
                "type": "agent_reference"  # sem versão = usa a mais recente
            }
        },
        input=texto
    )
    return response.output_text


# ------------------------------------------------------------------
# 6. Interface
# ------------------------------------------------------------------
st.title("🦇 Batman Agent · Microsoft Foundry")
st.caption("Converse com o Cavaleiro das Trevas: um agente do Foundry Agent Service com memória por conversa.")

# Sem as variáveis do .env não há como conectar: avisa e para o app
if not FOUNDRY_PROJECT_ENDPOINT or not MODEL_DEPLOYMENT_NAME:
    st.error("Configure FOUNDRY_PROJECT_ENDPOINT e MODEL_DEPLOYMENT_NAME no arquivo `.env`.")
    st.stop()

# Cria a primeira conversa só uma vez por sessão do navegador
if "conversation_id" not in st.session_state:
    try:
        nova_conversa()
    except ClientAuthenticationError:
        st.error("Falha no login da Azure. Execute `az login` no terminal e recarregue a página.")
        st.stop()
    except (HttpResponseError, APIError) as erro:
        st.error(f"Erro ao criar a conversa no Foundry: {erro}")
        st.stop()

with st.sidebar:
    st.header("⚙️ Agente")
    st.caption(f"Agente: `{AGENT_NAME}`  \nModelo: `{MODEL_DEPLOYMENT_NAME}`")
    st.caption(f"Conversa: `{st.session_state.conversation_id}`")

    # Nova conversa = o agente "esquece" tudo o que foi dito antes
    if st.button("🗑️ Nova conversa"):
        nova_conversa()
        st.rerun()

    st.divider()
    with st.expander("📝 Publicar nova versão do agente"):
        instrucoes = st.text_area("Instruções (system prompt)", value=INSTRUCOES_PADRAO, height=150)
        if st.button("🚀 Publicar versão"):
            try:
                agente = publicar_agente(instrucoes)
                st.success(f"Versão {agente.version} publicada!")
            except (HttpResponseError, ClientAuthenticationError) as erro:
                st.error(f"Erro ao publicar o agente: {erro}")

# Reexibe o histórico da conversa
for message in st.session_state.messages:
    with st.chat_message(message["role"], avatar="🦇" if message["role"] == "assistant" else None):
        st.markdown(message["content"])

if prompt := st.chat_input("Fale com o Batman"):
    # Usuário
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Agente
    with st.chat_message("assistant", avatar="🦇"):
        with st.spinner("O Batman está analisando o caso..."):
            try:
                resposta = perguntar(prompt)
            except ClientAuthenticationError:
                st.error("Falha no login da Azure. Execute `az login` no terminal.")
            except (HttpResponseError, APIError) as erro:
                st.error(f"Erro ao consultar o agente: {erro}")
            else:
                st.markdown(resposta)
                st.session_state.messages.append({"role": "assistant", "content": resposta})
